In [ ]:
import os
import sys
import gc
import time
import argparse
import logging
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple, Union
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
)

In [ ]:
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
CLASS_NAMES = [
    "planet",
    "eclipsing_binary",
    "background_binary",
    "pulsating_star",
    "flare_star",
    "starspots",
    "instrument_glitch",
    "quiet_star",
]

In [ ]:
def setup_logger(log_file_path: Optional[Path] = None) -> logging.Logger:
    """Configures a dedicated logger that writes to stdout and an evaluation file."""
    logger = logging.getLogger("CentroidEvaluationPipeline")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    formatter = logging.Formatter("%(asctime)s [%(levelname)s] %(message)s")
    # Console Handler
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)
    ch.setFormatter(formatter)
    logger.addHandler(ch)
    # File Handler
    if log_file_path:
        log_file_path.parent.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(str(log_file_path), mode="w", encoding="utf-8")
        fh.setLevel(logging.INFO)
        fh.setFormatter(formatter)
        logger.addHandler(fh)
    return logger

In [ ]:
def resolve_data_directory(explicit_dir: Optional[Union[str, Path]] = None) -> Path:
    """
    Intelligently discovers the data directory across local workspaces and Colab.
    Checks explicit parameter, CWD, Colab Drive mounts, and script location.
    """
    if explicit_dir:
        cand = Path(explicit_dir).resolve()
        if cand.exists() and (cand / "train_centroid.npy").exists():
            return cand
    candidates = [
        Path.cwd(),
        Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd(),
        Path("/content/drive/MyDrive/Hack4Hack2026"),
        Path("/content/drive/MyDrive/Hack4Hack"),
        Path("/content/drive/MyDrive"),
        Path("/content"),
    ]
    for candidate in candidates:
        if candidate.exists() and (candidate / "train_centroid.npy").exists():
            return candidate
    return Path.cwd()

In [ ]:
def extract_centroid_features(
    centroid_array: np.ndarray,
    chunk_size: int = 3000,
    logger: Optional[logging.Logger] = None,
) -> pd.DataFrame:
    """
    Vectorized, memory-efficient feature extraction for spatial centroid time series.
    Designed specifically to detect background binaries and pointing glitches.
    Parameters:
    -----------
    centroid_array : np.ndarray
        Shape (N_stars, 2, 1315), float16 or float32.
        Index 0 along axis 1 corresponds to Column (X) offset in millipixels.
        Index 1 along axis 1 corresponds to Row (Y) offset in millipixels.
    chunk_size : int
        Number of stars processed per vectorized chunk to conserve RAM.
    logger : logging.Logger
        Optional logger instance.

    Returns:
    --------
    pd.DataFrame : Extracted centroid feature matrix with N_stars rows.
    """
    n_stars, n_dims, n_cadences = centroid_array.shape
    if n_dims != 2:
        raise ValueError(f"Expected centroid array with 2 dimensions (col, row), got shape {centroid_array.shape}")
    if logger:
        logger.info(f"Extracting spatial centroid stability features for {n_stars:,} stars ({n_cadences} cadences)...")
    feature_chunks = []
    for start_idx in range(0, n_stars, chunk_size):
        end_idx = min(start_idx + chunk_size, n_stars)
        # Load slice as float32 for high numerical precision without excessive memory
        chunk = centroid_array[start_idx:end_idx].astype(np.float32)
        col = chunk[:, 0, :]  # Shape: (B, 1315) - Column (X) millipixels
        row = chunk[:, 1, :]  # Shape: (B, 1315) - Row (Y) millipixels
        # 1. Basic Dispersions and Positions
        col_mean = np.nanmean(col, axis=1)
        row_mean = np.nanmean(row, axis=1)
        col_std = np.nanstd(col, axis=1)
        row_std = np.nanstd(row, axis=1)
        col_med = np.nanmedian(col, axis=1)
        row_med = np.nanmedian(row, axis=1)

        # 2. Peak-to-Peak Movements & Centroid Shift Offsets
        col_min = np.nanmin(col, axis=1)
        col_max = np.nanmax(col, axis=1)
        row_min = np.nanmin(row, axis=1)
        row_max = np.nanmax(row, axis=1)
        centroid_offset_x = col_max - col_min
        centroid_offset_y = row_max - row_min
        col_ptp = centroid_offset_x
        row_ptp = centroid_offset_y
        # Robust P95 - P5 range (filters out single-cadence cosmic ray blips)
        col_p05, col_p95 = np.nanpercentile(col, [5, 95], axis=1)
        row_p05, row_p95 = np.nanpercentile(row, [5, 95], axis=1)
        col_p95_p5_range = col_p95 - col_p05
        row_p95_p5_range = row_p95 - row_p05

        # 3. Robust Non-parametric Motion Metrics (handling NaNs and non-Gaussianity)
        col_q25, col_q75 = np.nanpercentile(col, [25, 75], axis=1)
        row_q25, row_q75 = np.nanpercentile(row, [25, 75], axis=1)
        col_iqr = col_q75 - col_q25
        row_iqr = row_q75 - row_q25
        # Median Absolute Deviation (MAD)
        col_mad = np.nanmedian(np.abs(col - col_med[:, None]), axis=1)
        row_mad = np.nanmedian(np.abs(row - row_med[:, None]), axis=1)
        # Higher statistical moments: Skewness and Kurtosis
        col_skew = stats.skew(col, axis=1, nan_policy="omit")
        row_skew = stats.skew(row, axis=1, nan_policy="omit")
        col_kurt = stats.kurtosis(col, axis=1, nan_policy="omit")
        row_kurt = stats.kurtosis(row, axis=1, nan_policy="omit")

        # 4. 2D Euclidean Distance / Radial Displacement from Median Target Center
        dx = col - col_med[:, None]
        dy = row - row_med[:, None]
        radial_r = np.sqrt(dx**2 + dy**2)
        radial_dist_mean = np.nanmean(radial_r, axis=1)
        radial_dist_std = np.nanstd(radial_r, axis=1)
        radial_dist_med = np.nanmedian(radial_r, axis=1)
        radial_dist_max = np.nanmax(radial_r, axis=1)
        radial_dist_q95 = np.nanpercentile(radial_r, 95, axis=1)
        radial_r_q25, radial_r_q75 = np.nanpercentile(radial_r, [25, 75], axis=1)
        radial_dist_iqr = radial_r_q75 - radial_r_q25
        total_ptp_displacement = np.sqrt(centroid_offset_x**2 + centroid_offset_y**2)
        # 5. Cadence-to-Cadence Velocity / Jumps (Catches pointing glitches and abrupt shifts)
        diff_col = np.diff(col, axis=1)
        diff_row = np.diff(row, axis=1)
        diff_radial = np.sqrt(diff_col**2 + diff_row**2)
        col_diff_std = np.nanstd(diff_col, axis=1)
        row_diff_std = np.nanstd(diff_row, axis=1)
        col_diff_max = np.nanmax(np.abs(diff_col), axis=1)
        row_diff_max = np.nanmax(np.abs(diff_row), axis=1)
        radial_jump_max = np.nanmax(diff_radial, axis=1)

        # 6. Spatial Covariance & Correlation (Directional drift / Astrometric shift)
        cov_xy = np.nanmean(dx * dy, axis=1)
        denom = (col_std * row_std) + 1e-9
        col_row_corr = np.where((col_std > 1e-6) & (row_std > 1e-6), cov_xy / denom, 0.0)
        # 7. Quality & Data Gap Metrics
        nan_count = np.isnan(col).sum(axis=1)
        nan_fraction = nan_count / float(n_cadences)
        chunk_dict = {
            # Required primary stability metrics
            "col_std": col_std,
            "row_std": row_std,
            "centroid_offset_x": centroid_offset_x,
            "centroid_offset_y": centroid_offset_y,
            "col_ptp": col_ptp,
            "row_ptp": row_ptp,
            # Position centers
            "col_mean": col_mean,
            "row_mean": row_mean,
            "col_median": col_med,
            "row_median": row_med,

            # Robust dispersion metrics
            "col_iqr": col_iqr,
            "row_iqr": row_iqr,
            "col_mad": col_mad,
            "row_mad": row_mad,
            "col_skew": col_skew,
            "row_skew": row_skew,
            "col_kurt": col_kurt,
            "row_kurt": row_kurt,
            "col_p95_p5_range": col_p95_p5_range,
            "row_p95_p5_range": row_p95_p5_range,
            # 2D Euclidean / Radial Displacement
            "radial_dist_mean": radial_dist_mean,
            "radial_dist_std": radial_dist_std,
            "radial_dist_median": radial_dist_med,
            "radial_dist_max": radial_dist_max,
            "radial_dist_q95": radial_dist_q95,
            "radial_dist_iqr": radial_dist_iqr,
            "total_ptp_displacement": total_ptp_displacement,

            # Dynamics & Jumps
            "col_diff_std": col_diff_std,
            "row_diff_std": row_diff_std,
            "col_diff_max": col_diff_max,
            "row_diff_max": row_diff_max,
            "radial_jump_max": radial_jump_max,
            # Spatial Correlation
            "col_row_cov": cov_xy,
            "col_row_corr": col_row_corr,
            # Quality
            "centroid_nan_count": nan_count,
            "centroid_nan_fraction": nan_fraction,
        }
        chunk_df = pd.DataFrame(chunk_dict)
        feature_chunks.append(chunk_df)
        del chunk, col, row, dx, dy, radial_r, diff_col, diff_row, diff_radial
        gc.collect()


    features_df = pd.concat(feature_chunks, axis=0, ignore_index=True)
    # Impute any edge-case residual NaNs/Infs
    features_df = features_df.replace([np.inf, -np.inf], np.nan)
    features_df = features_df.fillna(features_df.median())
    features_df = features_df.fillna(0.0)
    return features_df

def get_baseline_classifier(random_state: int = 42):

    try:
        import lightgbm as lgb
        model = lgb.LGBMClassifier(
            objective="multiclass",
            num_class=8,
            n_estimators=150,
            learning_rate=0.08,
            num_leaves=31,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        )
        model_name = "LightGBM (LGBMClassifier)"
    except ImportError:
        from sklearn.ensemble import HistGradientBoostingClassifier
        model = HistGradientBoostingClassifier(
            max_iter=120,
            learning_rate=0.08,
            max_leaf_nodes=31,
            random_state=random_state,
        )
        model_name = "sklearn (HistGradientBoostingClassifier)"
    return model, model_name

def run_cross_validation(
    X: pd.DataFrame,
    y: np.ndarray,
    label_encoder: LabelEncoder,
    n_splits: int = 5,
    seed: int = 42,
    logger: Optional[logging.Logger] = None,
) -> Tuple[np.ndarray, float, str]:
    """
    Executes a 5-Fold Stratified Cross-Validation loop, tracks Macro-averaged F1,
    and produces detailed per-class performance reports.
    Returns:
    --------
    oof_predictions : np.ndarray of shape (N,)
    oof_macro_f1    : float
    full_report_str : str
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    n_classes = len(label_encoder.classes_)
    oof_pred_probs = np.zeros((len(X), n_classes), dtype=np.float32)
    fold_scores = []
    log_lines = []
    def log(msg: str):
        if logger:
            logger.info(msg)
        log_lines.append(msg)
    base_model, model_name = get_baseline_classifier(random_state=seed)
    log("=" * 80)
    log(f"STARTING {n_splits}-FOLD STRATIFIED CROSS-VALIDATION")
    log(f"Model Engine: {model_name}")
    log(f"Training Samples: {len(X):,} | Features: {X.shape[1]}")
    log(f"Target Classes ({n_classes}): {list(label_encoder.classes_)}")
    log("=" * 80)
    start_time = time.time()
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        fold_start = time.time()
        X_train, y_train = X.iloc[train_idx], y[train_idx]
        X_val, y_val = X.iloc[val_idx], y[val_idx]
        model, _ = get_baseline_classifier(random_state=seed + fold)
        model.fit(X_train, y_train)
        # Predict probabilities
        val_probs = model.predict_proba(X_val)
        oof_pred_probs[val_idx] = val_probs
        val_preds = np.argmax(val_probs, axis=1)
        fold_macro_f1 = f1_score(y_val, val_preds, average="macro")
        fold_acc = accuracy_score(y_val, val_preds)
        fold_scores.append(fold_macro_f1)
        elapsed = time.time() - fold_start
        log(f"Fold {fold}/{n_splits} | Macro F1: {fold_macro_f1:.4f} | Accuracy: {fold_acc:.4f} | Time: {elapsed:.1f}s")

    oof_preds = np.argmax(oof_pred_probs, axis=1)
    overall_macro_f1 = f1_score(y, oof_preds, average="macro")
    overall_acc = accuracy_score(y, oof_preds)
    total_elapsed = time.time() - start_time
    log("-" * 80)
    log(f"OVERALL OUT-OF-FOLD EVALUATION SUMMARY (Total CV Time: {total_elapsed:.1f}s)")
    log(f"PRIMARY COMPETITION METRIC -> Macro-averaged F1: {overall_macro_f1:.4f}")
    log(f"Fold F1 Mean: {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}")
    log(f"Overall Accuracy: {overall_acc:.4f}")
    log("-" * 80)

    # Detailed per-class classification report
    report = classification_report(
        y,
        oof_preds,
        target_names=label_encoder.classes_,
        digits=4,
    )
    log("DETAILED PER-CLASS PERFORMANCE REPORT:")
    log(report)
    # Confusion matrix breakdown
    cm = confusion_matrix(y, oof_preds)
    cm_df = pd.DataFrame(cm, index=label_encoder.classes_, columns=label_encoder.classes_)
    log("CONFUSION MATRIX (Actual rows vs Predicted columns):")
    log("\n" + cm_df.to_string())

    # Highlight background_binary performance (key target of centroid analysis)
    if "background_binary" in label_encoder.classes_:
        bg_idx = list(label_encoder.classes_).index("background_binary")
        bg_prec = cm[bg_idx, bg_idx] / max(1, cm[:, bg_idx].sum())
        bg_rec = cm[bg_idx, bg_idx] / max(1, cm[bg_idx, :].sum())
        bg_f1 = (2 * bg_prec * bg_rec) / max(1e-9, bg_prec + bg_rec)
        log("-" * 80)
        log(f"CRITICAL FOCUS: 'background_binary' Metrics -> Precision: {bg_prec:.4f} | Recall: {bg_rec:.4f} | F1: {bg_f1:.4f}")
        log("-" * 80)
    full_report_str = "\n".join(log_lines)
    return oof_preds, overall_macro_f1, full_report_str

def run_centroid_pipeline(
    data_dir: Optional[Union[str, Path]] = None,
    output_dir: Optional[Union[str, Path]] = None,
    n_splits: int = 5,
    seed: int = 42,
    chunk_size: int = 3000,
) -> Dict[str, Any]:
    """
    Full end-to-end execution of Centroid Analysis & Model Evaluation.
    Produces train_centroid_features.csv, test_centroid_features.csv, and model_evaluation_log.txt.
    """
    base_dir = resolve_data_directory(data_dir)
    out_dir = Path(output_dir).resolve() if output_dir else base_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    log_file = out_dir / "model_evaluation_log.txt"
    logger = setup_logger(log_file)
    logger.info("=" * 80)
    logger.info("PLANET HUNTERS 2026 - CENTROID ANALYSIS & EVALUATION PIPELINE")
    logger.info(f"Data Source Directory : {base_dir}")
    logger.info(f"Output Directory      : {out_dir}")
    logger.info("=" * 80)
    # 1. Verify existence of required raw files
    train_cen_path = base_dir / "train_centroid.npy"
    test_cen_path = base_dir / "test_centroid.npy"
    train_stars_path = base_dir / "train_stars.csv"
    test_stars_path = base_dir / "test_stars.csv"
    for p in [train_cen_path, test_cen_path, train_stars_path, test_stars_path]:
        if not p.exists():
            msg = f"Critical file missing: {p.name} not found in {base_dir}"
            logger.error(msg)
            raise FileNotFoundError(msg)
    # 2. Load Metadata Tables
    logger.info("Loading train_stars.csv and test_stars.csv...")
    train_stars = pd.read_csv(train_stars_path)
    test_stars = pd.read_csv(test_stars_path)
    logger.info(f"Loaded {len(train_stars):,} train stars and {len(test_stars):,} test stars.")
    # 3. Memory-efficient Loading and Feature Extraction: Training Centroids
    logger.info("Loading train_centroid.npy via memory mapping (mmap_mode='r')...")
    train_cen_mmap = np.load(train_cen_path, mmap_mode="r")
    logger.info(f"Train centroid shape: {train_cen_mmap.shape}, dtype: {train_cen_mmap.dtype}")
    train_features = extract_centroid_features(
        centroid_array=train_cen_mmap,
        chunk_size=chunk_size,
        logger=logger,
    )
    # Insert primary key 'id' and 'label' for seamless downstream merging
    train_features.insert(0, "id", train_stars["id"].values)
    train_features["label"] = train_stars["label"].values
    train_out_path = out_dir / "train_centroid_features.csv"
    train_features.to_csv(train_out_path, index=False)
    logger.info(f"Saved independent train features to: {train_out_path.name} ({train_features.shape[0]} rows, {train_features.shape[1]} cols)")
    # 4. Memory-efficient Loading and Feature Extraction: Test Centroids
    logger.info("Loading test_centroid.npy via memory mapping (mmap_mode='r')...")
    test_cen_mmap = np.load(test_cen_path, mmap_mode="r")
    logger.info(f"Test centroid shape: {test_cen_mmap.shape}, dtype: {test_cen_mmap.dtype}")
    test_features = extract_centroid_features(
        centroid_array=test_cen_mmap,
        chunk_size=chunk_size,
        logger=logger,
    )
    # Insert primary key 'id' for lossless merge
    test_features.insert(0, "id", test_stars["id"].values)
    test_out_path = out_dir / "test_centroid_features.csv"
    test_features.to_csv(test_out_path, index=False)
    logger.info(f"Saved independent test features to: {test_out_path.name} ({test_features.shape[0]} rows, {test_features.shape[1]} cols)")
    # Free memory
    del train_cen_mmap, test_cen_mmap
    gc.collect()
    # 5. Prepare Modeling Dataset
    logger.info("Preparing feature matrix for Stratified 5-Fold Cross-Validation...")
    feature_cols = [c for c in train_features.columns if c not in ["id", "label"]]
    X = train_features[feature_cols].copy()
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(train_features["label"].values)
    # 6. Run 5-Fold Stratified Cross-Validation Loop & Metric Tracking
    oof_preds, macro_f1, full_report = run_cross_validation(
        X=X,
        y=y,
        label_encoder=label_encoder,
        n_splits=n_splits,
        seed=seed,
        logger=logger,
    )
    logger.info("=" * 80)
    logger.info("PIPELINE EXECUTION COMPLETE!")
    logger.info(f"1. Independent Train Matrix : {train_out_path.resolve()}")
    logger.info(f"2. Independent Test Matrix  : {test_out_path.resolve()}")
    logger.info(f"3. Full Evaluation Log      : {log_file.resolve()}")
    logger.info(f"Final Out-of-Fold Macro F1  : {macro_f1:.4f}")
    logger.info("All teammate scripts and directories preserved without modifications.")
    logger.info("=" * 80)
    return {
        "macro_f1": macro_f1,
        "train_features_path": str(train_out_path),
        "test_features_path": str(test_out_path),
        "log_path": str(log_file),
    }
if __name__ == "__main__":
    from typing import Any
    parser = argparse.ArgumentParser(
        description="Planet Hunters Centroid Analysis & Cross-Validation Evaluation Pipeline."
    )
    parser.add_argument(
        "--data_dir",
        type=str,
        default=None,
        help="Path to folder containing train_centroid.npy, test_centroid.npy, and *_stars.csv",
    )
    parser.add_argument(
        "--output_dir",
        type=str,
        default=None,
        help="Path where feature CSVs and evaluation log will be written",
    )
    parser.add_argument(
        "--n_splits",
        type=int,
        default=5,
        help="Number of Stratified K-Fold CV splits (default: 5)",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
        help="Random seed for reproducibility (default: 42)",
    )
    parser.add_argument(
        "--chunk_size",
        type=int,
        default=3000,
        help="Batch chunk size for memory-efficient feature computation (default: 3000)",
    )
    args, unknown = parser.parse_known_args()
    run_centroid_pipeline(
        data_dir=args.data_dir,
        output_dir=args.output_dir,
        n_splits=args.n_splits,
        seed=args.seed,
        chunk_size=args.chunk_size,
    )

2026-10-22 19:26:43,964 [INFO] ================================================================================


INFO:CentroidEvaluationPipeline:================================================================================


2026-10-22 19:26:43,967 [INFO] PLANET HUNTERS 2026 - CENTROID ANALYSIS & EVALUATION PIPELINE


INFO:CentroidEvaluationPipeline:PLANET HUNTERS 2026 - CENTROID ANALYSIS & EVALUATION PIPELINE


2026-10-22 19:26:43,972 [INFO] Data Source Directory : /content


INFO:CentroidEvaluationPipeline:Data Source Directory : /content


... (3 outputs left out)

INFO:CentroidEvaluationPipeline:================================================================================


2026-10-22 19:26:43,986 [ERROR] Critical file missing: train_centroid.npy not found in /content


ERROR:CentroidEvaluationPipeline:Critical file missing: train_centroid.npy not found in /content


FileNotFoundError: Critical file missing: train_centroid.npy not found in /content

In [ ]:
import os

required_files = [
    "train_centroid.npy",
    "test_centroid.npy",
    "train_stars.csv",
    "test_stars.csv"
]

for file in required_files:
    if os.path.exists("/content/" + file):
        print("✅ Found:", file)
    else:
        print("❌ Missing:", file)